# Tracing Lab

Tracing is how you watch a simulation run without touching the steering wheel. In this lab we will capture logs, read them as records, attach them to a timeline, and compare runs with tracing on and off.


## 1. Start With One Log Record

A trace record is a small fact about the run: when it happened, what event it belongs to, what action it describes, and any extra context worth keeping.


In [ ]:
from simyuj.tracing import LogLevel, MemorySink, SimulationLogger

sink = MemorySink()
logger = SimulationLogger(level=LogLevel.INFO, sinks=[sink], session_id="trace-lab")

record = logger.info(
    "lab.link",
    "fiber segment measured",
    sim_time=12_500,
    link_id="alice-bob",
    meta={"distance_km": 25, "loss_db": 5.0},
)

print("records stored:", len(sink.records))
print("first category:", sink.records[0].category)
print("first message:", sink.records[0].message)
print("first level:", sink.records[0].level.name)
print("first sequence:", sink.records[0].sequence)


In [ ]:
first = sink.records[0]

print("sim_time:", first.sim_time, "ps")
print("link_id:", first.link_id)
print("session_id:", first.session_id)
print("meta:", first.meta)


## 2. Levels Decide How Much Noise You Want

`INFO` is good for a normal run. `TRACE` is for opening the engine hood and watching schedule/execution details.


In [ ]:
for level in LogLevel:
    print(f"{level.name:7s}", int(level))


In [ ]:
sink.clear()
logger.level = LogLevel.INFO

logger.error("lab.level", "serious problem")
logger.warning("lab.level", "worth attention")
logger.info("lab.level", "normal progress")
logger.debug("lab.level", "extra detail")
logger.trace("lab.level", "deep detail")

print("at INFO we kept:")
for item in sink.records:
    print(item.sequence, item.level.name, item.message)
print("next sequence:", logger.sequence)


In [ ]:
sink.clear()
logger.level = LogLevel.TRACE

logger.debug("lab.level", "debug detail is now visible")
logger.trace("lab.level", "trace detail is now visible")

print("at TRACE we kept:")
for item in sink.records:
    print(item.sequence, item.level.name, item.message)


In [ ]:
sink.clear()
logger.level = LogLevel.OFF

logger.error("lab.level", "even this is quiet")
logger.info("lab.level", "also quiet")

print("records while OFF:", len(sink.records))
print("next sequence stayed:", logger.sequence)


## 3. TextSink Is For Human Eyes

`MemorySink` is best when you want to inspect records in code. `TextSink` turns the same records into compact lines.


In [ ]:
from io import StringIO
from simyuj.tracing import TextSink

stream = StringIO()
text_logger = SimulationLogger(
    level=LogLevel.INFO,
    sinks=[TextSink(stream=stream)],
    session_id="text-run",
)

text_logger.info(
    "lab.channel",
    "classical message delivered",
    sim_time=8_000,
    event_id=4,
    action="CLASSICAL_ARRIVE",
    source_name="Alice",
    target_name="Bob",
    meta={"distance_km": 12, "delay_ns": 60},
)

print(stream.getvalue())


## 4. JSONL Is For Runs You Want To Keep

Files should be deliberate. This cell writes one temporary JSONL file, reads the line back, and removes the temporary area when the cell ends.


In [ ]:
import json
from tempfile import TemporaryDirectory
from pathlib import Path
from simyuj.tracing import JsonlSink

with TemporaryDirectory() as tmp:
    path = Path(tmp) / "trace.jsonl"
    with JsonlSink(path=path, auto_flush=True) as json_sink:
        json_logger = SimulationLogger(
            level=LogLevel.TRACE,
            sinks=[json_sink],
            session_id="json-run",
        )
        json_logger.info(
            "lab.source",
            "photon emitted",
            sim_time=3_000,
            event_id=9,
            action="SOURCE_EMIT",
            node_id="alice",
            meta={"wavelength_nm": 1550, "attempt": 1},
        )

    payload = json.loads(path.read_text(encoding="utf-8").strip())
    print("json keys:", sorted(payload))
    print("category:", payload["category"])
    print("meta:", payload["meta"])


## 5. Put A Logger On A Timeline

The timeline already knows the current time, event ids, action names, and target names. At `TRACE`, it records scheduling and execution around your components.


In [ ]:
from simyuj.engine import Component, Event, Timeline

class LabTarget(Component):
    def __init__(self, name):
        self.name = name
        self.seen = []

    def handle_event(self, event, timeline):
        self.seen.append((timeline.current_time, event.event_id, event.action))
        timeline.log(
            LogLevel.INFO,
            "lab.target",
            f"{self.name} handled {event.action}",
            event=event,
            node_id=self.name,
            meta={"queue_note": "component work"},
        )

        if event.action == "PING":
            timeline.schedule(
                Event(
                    time=timeline.current_time + 40,
                    target_ref=self,
                    action="PONG",
                    payload_ref={"round": 1},
                    source=self,
                )
            )


In [ ]:
timeline_sink = MemorySink()
timeline_logger = SimulationLogger(level=LogLevel.TRACE, sinks=[timeline_sink])
timeline = Timeline(master_seed=7, logger=timeline_logger)
target = LabTarget("node-a")

timeline.schedule(
    Event(time=100, target_ref=target, action="PING", payload_ref=None)
)
summaries = timeline.run_until_empty()

print("target saw:", target.seen)
print("batch times:", [summary.batch_time for summary in summaries])
print("events executed:", [summary.event_ids for summary in summaries])


In [ ]:
print("timeline records:")
for item in timeline_sink.records:
    print(
        item.sequence,
        item.level.name,
        item.category,
        "t=", item.sim_time,
        "event=", item.event_id,
        "action=", item.action,
    )


In [ ]:
print("records from our component:")
for item in timeline_sink.records:
    if item.category == "lab.target":
        print(item.message)
        print("  target:", item.target_name)
        print("  node:", item.node_id)
        print("  meta:", item.meta)


## 6. INFO Versus TRACE On The Same Timeline

Use `INFO` when you want lifecycle and component notes. Use `TRACE` when schedule order or event execution details matter.


In [ ]:
from collections import Counter


def run_ping_pong(level):
    sink = MemorySink()
    logger = SimulationLogger(level=level, sinks=[sink])
    timeline = Timeline(master_seed=7, logger=logger)
    target = LabTarget("node-a")
    timeline.schedule(Event(time=100, target_ref=target, action="PING", payload_ref=None))
    summaries = timeline.run_until_empty()
    return target.seen, summaries, sink.records

info_seen, info_summaries, info_records = run_ping_pong(LogLevel.INFO)
trace_seen, trace_summaries, trace_records = run_ping_pong(LogLevel.TRACE)

print("same component view:", info_seen == trace_seen)
print("INFO categories:", [item.category for item in info_records])
print("TRACE category counts:", dict(Counter(item.category for item in trace_records)))


## 7. Read A Trace Like A Flight Recorder

When something looks wrong, group records by category first. Then zoom into the event id or node that looks suspicious.


In [ ]:
from collections import Counter

counts = Counter(item.category for item in trace_records)
for category, count in counts.items():
    print(f"{category:36s}", count)


In [ ]:
event_to_follow = trace_seen[0][1]
print("following event_id:", event_to_follow)

for item in trace_records:
    if item.event_id == event_to_follow:
        print(
            item.sequence,
            item.category,
            "t=", item.sim_time,
            "action=", item.action,
            "message=", item.message,
        )


## 8. Tracing Must Not Change The Run

This is the habit worth keeping: compare the observables, not the number of log records. Logs are notes about execution; they are not another event source.


In [ ]:
def fiber_decision(timeline, event, *, distance_km, attenuation_db_per_km):
    loss_db = distance_km * attenuation_db_per_km
    keep_probability = 10 ** (-loss_db / 10)
    sample = timeline.rng("fiber-hop").random()
    delivered = sample < keep_probability

    timeline.log(
        LogLevel.DEBUG,
        "lab.fiber",
        "loss sampled",
        event=event,
        link_id="alice-bob",
        meta={
            "distance_km": distance_km,
            "loss_db": round(loss_db, 3),
            "keep_probability": round(keep_probability, 4),
            "sample": round(sample, 4),
            "delivered": delivered,
        },
    )
    return delivered


In [ ]:
class LossyHop(Component):
    def __init__(self, detector, *, distance_km, attenuation_db_per_km):
        self.detector = detector
        self.distance_km = distance_km
        self.attenuation_db_per_km = attenuation_db_per_km

    def handle_event(self, event, timeline):
        delivered = fiber_decision(
            timeline,
            event,
            distance_km=self.distance_km,
            attenuation_db_per_km=self.attenuation_db_per_km,
        )
        if delivered:
            timeline.schedule(
                Event(
                    time=timeline.current_time + 250,
                    target_ref=self.detector,
                    action="DETECT",
                    payload_ref=event.payload_ref,
                    source=self,
                )
            )


In [ ]:
class Detector(Component):
    def __init__(self):
        self.clicks = []

    def handle_event(self, event, timeline):
        self.clicks.append((timeline.current_time, event.payload_ref["pulse_id"]))
        timeline.log(
            LogLevel.INFO,
            "lab.detector",
            "detector clicked",
            event=event,
            node_id="bob",
            meta={"pulse_id": event.payload_ref["pulse_id"]},
        )


In [ ]:
def run_lossy_link(logger=None):
    timeline = Timeline(master_seed=123, logger=logger)
    timeline.rng("fiber-hop")
    detector = Detector()
    hop = LossyHop(detector, distance_km=18, attenuation_db_per_km=0.2)

    for pulse_id, time in enumerate([0, 100, 200, 300, 400], start=1):
        timeline.schedule(
            Event(
                time=time,
                target_ref=hop,
                action="PULSE",
                payload_ref={"pulse_id": pulse_id},
            )
        )

    summaries = timeline.run_until_empty()
    return detector.clicks, summaries, timeline.stats

plain_clicks, plain_summaries, plain_stats = run_lossy_link(logger=None)
logged_sink = MemorySink()
logged_logger = SimulationLogger(level=LogLevel.DEBUG, sinks=[logged_sink])
logged_clicks, logged_summaries, logged_stats = run_lossy_link(logger=logged_logger)

print("plain clicks:", plain_clicks)
print("logged clicks:", logged_clicks)
print("same clicks:", plain_clicks == logged_clicks)
print("same stats:", plain_stats == logged_stats)
print("records captured:", len(logged_sink.records))


In [ ]:
print("loss samples:")
for item in logged_sink.records:
    if item.category == "lab.fiber":
        meta = dict(item.meta)
        print(
            "pulse event",
            item.event_id,
            "sample",
            meta["sample"],
            "keep below",
            meta["keep_probability"],
            "delivered",
            meta["delivered"],
        )


In [ ]:
print("detector records:")
for item in logged_sink.records:
    if item.category == "lab.detector":
        print(item.sim_time, item.message, dict(item.meta))


## 9. A Small Trace Reading Exercise

Change the distance in `run_lossy_link`, move the logger between `INFO`, `DEBUG`, and `TRACE`, and watch what changes. The clicks should follow the physics and the seed; the trace should only change how much evidence you kept.


In [ ]:
experiment_sink = MemorySink()
experiment_logger = SimulationLogger(level=LogLevel.TRACE, sinks=[experiment_sink])
clicks, summaries, stats = run_lossy_link(logger=experiment_logger)

print("clicks:", clicks)
print("batch times:", [summary.batch_time for summary in summaries])
print("first six trace categories:")
for item in experiment_sink.records[:6]:
    print(item.sequence, item.category, item.action, item.sim_time)
